Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Cu(111)表面原子の溶出の自由エネルギー計算

![Graphical Abstract](./assets/movies/traj-movie-small.gif)

このExampleでは、**Cu(111)スラブ/水界面** において、Cu表面原子をz方向（表面垂直方向）に引き離す過程の自由エネルギープロファイルをアンブレラサンプリングで計算するワークフローを紹介します。具体的には、PLUMEDのMOVINGRESTRAINTによるSteered MDで反応座標空間を探索し、各ウィンドウでアンブレラサンプリングを実施した後、MBAR法を用いて自由エネルギーを算出します。

## Notebookの概要

計算は以下の6つのステップ（ノートブック）に分かれています。順番に実行することで、モデリングから解析までを一通り行うことができます。
なお、`assets`ディレクトリにそれぞれのノートブックの実行に必要なインプットを準備しているため、独立に実施することも可能です。

| Step | Notebook | 概要 |
| :--- | :--- | :--- |
| **01** | [01_modeling_cu_water_interface_ja.ipynb](./01_modeling_cu_water_interface_ja.ipynb) | **モデリング**<br>ASEとpfcc_extrasを使用して、Cu(111)スラブと水分子の界面構造を作成・最適化します。 |
| **02** | [02_equilibrium_npzt_md_ja.ipynb](./02_equilibrium_npzt_md_ja.ipynb) | **平衡化MD**<br>NPzTアンサンブル（z方向のみ圧力制御）で界面構造を375 Kで平衡化します。 |
| **03** | [03_steered_md_ja.ipynb](./03_steered_md_ja.ipynb) | **Steered MD (SMD)**<br>PLUMEDのMOVINGRESTRAINTを使用して、Cu表面原子をz方向に引き離し、反応座標全体をカバーするトラジェクトリを作成します。 |
| **04** | [04_select_umbrella_sampling_initial_structures_ja.ipynb](./04_select_umbrella_sampling_initial_structures_ja.ipynb) | **初期構造の選出**<br>SMDの軌跡から、各アンブレラサンプリング・ウィンドウ（反応座標の特定の値）に対応する構造を抽出します。 |
| **05** | [05_umbrella_sampling_ja.ipynb](./05_umbrella_sampling_ja.ipynb) | **アンブレラサンプリング**<br>抽出した構造を初期値として、調和ポテンシャルで拘束した多数のMDをjoblibで並列実行し、データを収集します。 |
| **06** | [06_mbar_free_energy_ja.ipynb](./06_mbar_free_energy_ja.ipynb) | **自由エネルギー解析**<br>得られたデータをMBAR法（pymbar）で解析し、Cu原子のz座標に対する自由エネルギープロファイル（PMF）を描画します。 |

## 対象システムと反応座標

* **系**: Cu(111)スラブ/水界面
    * Cu(111)スラブ: 6×8×6 (288原子)
    * 水分子: 250個 (750原子)
    * 合計: 1038原子、周期境界条件下でのシミュレーション
* **反応座標 (Collective Variable)**:
    * Cu表面原子1個（ASE index: 267、PLUMED index: 268）のZ座標
    * この位置を 10.2 Å から 18.0 Å 程度まで変化させます。

## 必要なライブラリ・環境設定について

このExampleを実行するには、以下のソフトウェア・パッケージが必要です。

* **PFP-API-CLIENT**: PFP (Preferred Potential) を使用するためのパッケージ。
* **ASE (Atomic Simulation Environment)**: 原子構造の操作やMDのインターフェースとして使用。
* **pfcc_extras**: LiquidGenerator（液体構造生成）、表面構造構築ツールなどを使用。
* **PLUMED**: Step03, Step05 のSteered MDやUmbrella Samplingで調和振動子による束縛を加えるのに使用。
    * ASEのPLUMED Calculatorインターフェースを使用します。
* **pymbar**: Step06 での自由エネルギー解析（MBAR法）に使用。
* **joblib**: Step05 でのアンブレラサンプリングの並列実行に使用。

### PLUMEDのInstall方法について
[PLUMED](https://www.plumed.org/)は、分子動力学シミュレーションパッケージと連携して、反応座標（集団変数）の計算やバイアス力の適用を行うことで、長時間を要する稀な現象の効率的なサンプリングや自由エネルギー解析を可能にするオープンソースライブラリとなっています。本ノートブックでは、反応座標 (ここではCu原子のz座標)に調和振動子を追加するのにPLUMEDを使用しているため、事前にPLUMEDをインストールする必要があります。

PLUMEDのインストール手順は以下の通りとなります。

#### 1. ソースコードのダウンロードとコンパイル
ターミナルで以下のコマンドを実行し、PLUMEDをビルドします。

```bash

# インストール用のディレクトリ
mkdir -p ~/local && cd ~/local

# PLUMEDのレポジトリをクローン
git clone https://github.com/plumed/plumed2.git plumed-2.9.0

# バージョンをv2.9.0にチェックアウト
cd plumed-2.9.0 && git checkout v2.9.0

# PLUMEDのビルド (configure & make)
./configure --disable-mpi --prefix=$HOME/local/plumed-2.9.0 && make -j"$(nproc)" && make install
```

#### 2. Pythonバインディングのインストール
ASEからPLUMEDを呼び出すためのPythonパッケージもインストールします。

```bash
$ pip install plumed
```



#### 注意事項:

ノートブック [03_steered_md_ja.ipynb](./03_steered_md_ja.ipynb) および [05_umbrella_sampling_ja.ipynb](./05_umbrella_sampling_ja.ipynb) の冒頭には、PLUMEDのインストールパスを指定する箇所があります（例：~/local/plumed-2.9.0）。 インストールしたバージョンやディレクトリ名（plumed-2.9.0 等）に合わせて、パスを適宜修正して実行してください。